In [35]:
using LowLevelFEM, LinearAlgebra

In [36]:
p = 3

3

In [37]:
structured_box_mesh(n=10, order=p)

openGeometry("box2.geo")
openPreProcessor()

mat = Material("body")
U = Field([mat], type=:VectorField, dim=3, field=:u);

In [38]:
k = mat.k

s = ScalarField(U, "right", (x, y, z)->y > 0.5 ? 1 : 10)

K = ∫(SymGrad(U) ⋅ [2 1 1 0 0 0; 1 2 1 0 0 0; 1 1 2 0 0 0; 0 0 0 1 0 0; 0 0 0 0 1 0; 0 0 0 0 0 1] ⋅ SymGrad(U))
f = ∫(U ⋅ [s, 0, 0], Γ="right")

bc = BoundaryCondition("left", ux=0, uy=0, uz=0);

In [39]:
fixed = constrainedDoFs(U, [bc])
free = freeDoFs(U, [bc]);

In [40]:
u1 = applyBoundaryConditions(U, [bc])
f_kin = K.A[:, fixed] * u1.a[fixed, 1]
u1.a[free] = (K.A[free, free]) \ (f.a[free, 1] - f_kin[free, 1]);

In [41]:
showDoFResults(u1, name="u1", visible=true);

In [42]:
T, R = reductionMatrices(U);

In [ ]:
u2 = applyBoundaryConditions(U, [bc])
f_kin = K.A[:, fixed] * u2.a[fixed, 1]
Kr = T[free, :]' * K.A[free, free] * T[free, :]
fr = T[free, :]' * (f.a[free, 1] - f_kin[free, 1])

ur = Kr \ fr

u2.a[free] = (T*ur)[free];

In [ ]:
showDoFResults(u2, name="u2", visible=true);

In [ ]:
norm(u2.a - u1.a) / norm(u1.a)

0.005181578955365662

In [ ]:
∫(U, "body", u1[1])

0.09407246104586142

In [ ]:
∫(U, "body", u2[1])

0.0936846099139728

In [ ]:
openPostProcessor();